#**MCQ CREATER APP**

## Install required libraries

In [1]:
# install dependencies

!pip install groq pypdf --quiet
!pip install --upgrade langchain langchain-core pinecone-client langchain-community langchain-text-splitters langchain-pinecone langchain-huggingface langchain_groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.

# import required libraries

# Define a flag to track if reinstallation was attempted
reinstalled_langchain_deps = False

try:
    # Attempt to import PyPDFDirectoryLoader to trigger the potential ImportError
    from langchain_community.document_loaders import PyPDFDirectoryLoader
except ImportError as e:
    # Check if the error is specifically related to BaseBlobParser or BaseLoader not found
    if "cannot import name 'BaseBlobParser'" in str(e) or "cannot import name 'BaseLoader'" in str(e):
        print("Detected LangChain version incompatibility (ImportError for 'BaseBlobParser' or 'BaseLoader').")
        print("Attempting to reinstall 'langchain-community' and 'langchain-core' to resolve the dependency conflict.")
        print("This will run 'pip uninstall' and 'pip install --upgrade' within this cell.")
        # Uninstall first to clear potentially conflicting versions
        # Use -y for non-interactive uninstall
        # !pip uninstall -y langchain-community langchain-core --quiet
        # Then reinstall/upgrade to fetch compatible latest versions
        # !pip install --upgrade langchain-community langchain-core --quiet
        print("Reinstallation attempt finished. Please re-run this cell to confirm the fix.")
        print("If the 'ImportError' persists after re-running, please try restarting the entire Colab runtime (Runtime -> Restart runtime...).")
        reinstalled_langchain_deps = True
    else:
        # If it's a different ImportError, re-raise it
        raise # Re-raise unexpected ImportErrors

# These imports are always needed. If reinstallation happened, PyPDFDirectoryLoader
# will be imported below. If no reinstallation, it was already successfully imported.
from pinecone import Pinecone
# Re-import PyPDFDirectoryLoader here in case reinstallation happened, or if it wasn't the initial problem
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

In [7]:
# Import the UserData module to access Colab secrets

from google.colab import userdata
from langchain_groq import ChatGroq

# Retrieve the API key from Colab secrets
GROQ_API_KEY = userdata.get('Groq_Api_key')

# Initialize the OpenAI client
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name='mixtral-8x7b-32768')

print("AI client initialized successfully!")

AI client initialized successfully!


## Cell 5 — PDF Upload & Text Extraction

**This cell defines all PDF-related utilities.**
- Supports uploading **PDF files** via the standard Colab file dialog
- Also supports reading PDFs from a file path already on disk
- Uses `pypdf` with `BytesIO` — no temp files, no disk writes needed
- Handles multi-page PDFs and skips unreadable pages gracefully

In [8]:
from google.colab import files

# upload pdf files

def upload_pdf_and_extract() -> bytes | None:

    print("Click the 'Choose Files' button below to upload your PDF...")
    print("Only .pdf files are accepted")
    print()

    # Open Colab file upload dialog
    uploaded = files.upload()

    if not uploaded:
        print("No file was uploaded. Please re-run this cell and select a PDF.")
        return None

    # Get filename (take first file if multiple selected)
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]

    # Validate: must be a PDF
    if not filename.lower().endswith(".pdf"):
        print(f"File '{filename}' is not a PDF.")
        print("   Please upload a .pdf file only.")
        return None

    print(f"Successfully uploaded '{filename}'.")
    return file_bytes

# Call the function to upload and extract file bytes
uploaded_file_bytes = upload_pdf_and_extract()

if uploaded_file_bytes:
    print(f"File bytes are ready for further processing. Size: {len(uploaded_file_bytes)} bytes.")

Click the 'Choose Files' button below to upload your PDF...
Only .pdf files are accepted



Saving Big Data Introduction.pdf to Big Data Introduction.pdf
Successfully uploaded 'Big Data Introduction.pdf'.
File bytes are ready for further processing. Size: 192342 bytes.


In [11]:
#extract txt from pdf file

import io
from pypdf import PdfReader # Import PdfReader
import re # Added to ensure 're' module is available

def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    reader = PdfReader(io.BytesIO(pdf_bytes))

    total_pages = len(reader.pages)
    if total_pages == 0:
        raise RuntimeError("PDF has no pages.")

    print(f"PDF has {total_pages} page(s). Extracting text...")

    all_text = []
    for page_num, page in enumerate(reader.pages):
        try:
            page_text = page.extract_text()
            if page_text and page_text.strip():
                all_text.append(page_text.strip())
        except Exception as e:
            print(f"Skipped page {page_num + 1} (unreadable): {e}")
    # The 'continue' statement was misplaced in the original code and has been removed.

    if not all_text:
        raise RuntimeError(
                "No text could be extracted from this PDF.\n"
                "This usually means the PDF contains scanned images only (no selectable text layer).\n"
                "Please use a PDF that was created digitally (e.g. exported from Word, Google Docs, etc.).")

    combined = "\n\n".join(all_text)
    # Assuming clean_text is defined globally in an earlier cell
    return clean_text(combined)

In [13]:
import re

# clean text

def clean_text(text: str) -> str:
    """
    Clean extracted PDF text:
    - Collapse multiple spaces/tabs into single space
    - Collapse 3+ newlines into 2 newlines
    - Remove very short lines (likely page numbers / headers)
    """
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    lines = [l.strip() for l in text.split('\n') if len(l.strip()) > 3]
    return '\n'.join(lines).strip()

In [14]:
# Extract text from the uploaded PDF bytes
if 'uploaded_file_bytes' in locals() and uploaded_file_bytes is not None:
    extracted_text = extract_text_from_pdf_bytes(uploaded_file_bytes)
    print(f"Successfully extracted {len(extracted_text)} characters from the PDF.")
else:
    print("No PDF bytes found to extract text from. Please run the upload cell first.")

PDF has 8 page(s). Extracting text...
Successfully extracted 11347 characters from the PDF.


In [15]:
import os

# extract text from pdf path
def extract_text_from_pdf_path(file_path: str) -> str:

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"PDF not found at: {file_path}")
    if not file_path.lower().endswith(".pdf"):
        raise ValueError(f"Expected a .pdf file, got: {file_path}")
    with open(file_path, "rb") as f:
        pdf_bytes = f.read()
    return extract_text_from_pdf_bytes(pdf_bytes)

In [16]:
if 'uploaded_file_bytes' in globals() and uploaded_file_bytes is not None:
    size_kb = len(uploaded_file_bytes) / 1024
    # filename is not globally available from the current setup of upload_pdf_and_extract.
    # The filename was printed in the upload_pdf_and_extract function itself.
    print(f" Uploaded file size: ({size_kb:.1f} KB)")
    print()
else:
    print("No uploaded file bytes found to report size.")

 Uploaded file size: (187.8 KB)



In [17]:
# Extract text

print(" Extracting text from PDF...")
try:
  extracted_text = extract_text_from_pdf_bytes(uploaded_file_bytes)
except RuntimeError as e:
  print(f"Extraction failed:\n   {e}")
except Exception as e:
  print(f" Unexpected error: {e}")

 Extracting text from PDF...
PDF has 8 page(s). Extracting text...


In [18]:
word_count = len(extracted_text.split())
char_count = len(extracted_text)

print("Extraction complete!")
print(f"{word_count:,} words  |  {char_count:,} characters")
print()
print("Preview (first 300 characters):")
print("   " + "─" * 52)

preview = extracted_text[:300].replace("\n", " ")
print(f"{preview}{'...' if len(extracted_text) > 300 else ''}")
print("   " + "─" * 52)

Extraction complete!
1,675 words  |  11,347 characters

Preview (first 300 characters):
   ────────────────────────────────────────────────────
Big Data What is Big Data? ● Big Data refers to extremely large datasets that are too complex and vast for traditional data processing tools handle effectively. These datasets come from various sources, like social media, sensors, business transactions, online activity, they generated increasing rat...
   ────────────────────────────────────────────────────


In [19]:
import json

def build_system_prompt(include_explanation: bool) -> str:
    """Builds the system prompt for the MCQ generation LLM."""
    explanation_directive = "Include a 'explanation' field for each answer." if include_explanation else ""
    return f"""You are an expert in creating multiple-choice questions from provided text. Your task is to generate MCQs based on the context given by the user. {explanation_directive}

Strictly output the response as a JSON array of objects. Each object must have the following keys: 'question', 'options' (an object with keys 'A', 'B', 'C', 'D'), 'correct_answer' (one of 'A', 'B', 'C', 'D').

Example JSON format:
[
  {{
    "question": "What is the capital of France?",
    "options": {{
      "A": "Berlin",
      "B": "Madrid",
      "C": "Paris",
      "D": "Rome"
    }},
    "correct_answer": "C",
    "explanation": "Paris is the capital and most populous city of France."
  }}
]"""

def build_user_prompt(text: str, num_mcqs: int, difficulty: str, subject: str) -> str:
    """Builds the user prompt for the MCQ generation LLM."""
    return f"""Generate {num_mcqs} multiple-choice questions on the subject of '{subject}' from the following text.

Difficulty: {difficulty}

Context:
{text}

Ensure questions are directly answerable from the provided text and options are plausible but distinct. The 'options' key must contain exactly A, B, C, D as keys."""

In [20]:
import json
import traceback # Import traceback for detailed error logging

def parse_mcq_json(raw: str) -> list:
    """
    Robustly parse MCQ JSON from the model response.
    Handles: bare array, wrapped object {mcqs: [...]}, markdown code fences.
    """
    raw = raw.strip()

    # Strip markdown fences if model adds them despite instructions
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    raw = raw.strip()

    parsed = None

    # Attempt 1: direct JSON parse
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Attempt 2: extract JSON array from text
    if parsed is None:
        arr_match = re.search(r'\[.*\]', raw, re.DOTALL)
        if arr_match:
            try:
                parsed = json.loads(arr_match.group())
            except json.JSONDecodeError:
                pass

    # Attempt 3: extract JSON object from text
    if parsed is None:
        obj_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if obj_match:
            try:
                parsed = json.loads(obj_match.group())
            except json.JSONDecodeError:
                pass

    if parsed is None:
        raise RuntimeError(f"Could not parse JSON from model response.\nRaw output (first 300 chars):\n{raw[:300]}")

    # Unwrap if model returned {"mcqs": [...]} or similar wrapper
    if isinstance(parsed, dict):
        for val in parsed.values():
            if isinstance(val, list):
                parsed = val
                break
        else:
            if "question" in parsed:
                parsed = [parsed]
            else:
                raise RuntimeError("Model returned a JSON object but no MCQ array found inside it.")

    if not isinstance(parsed, list):
        raise RuntimeError(f"Expected list of MCQs, got: {type(parsed)}")

    # Validate and clean each MCQ
    valid_mcqs = []
    for item in parsed:
        if not isinstance(item, dict):
            continue
        question = str(item.get("question", "")).strip()
        if not question:
            continue
        options = item.get("options", {})
        if not isinstance(options, dict):
            continue
        for label in ["A", "B", "C", "D"]:
            if label not in options or not str(options[label]).strip():
                options[label] = f"Option {label}"
        correct = str(item.get("correct_answer", "A")).strip().upper()
        if correct not in ["A", "B", "C", "D"]:
            correct = "A"
        explanation = str(item.get("explanation", "")).strip()
        valid_mcqs.append({
            "question": question,
            "options": {k: str(v).strip() for k, v in options.items() if k in ["A","B","C","D"]},
            "correct_answer": correct,
            "explanation": explanation
        })

    if not valid_mcqs:
        raise RuntimeError("No valid MCQs found in the parsed response.")
    return valid_mcqs


def generate_mcqs(text: str, num_mcqs: int, difficulty: str,
                  subject: str, include_explanation: bool, api_key: str) -> list:
    """
    Main function: calls Groq API and returns list of MCQ dicts.

    Parameters:
        text               : Source text to generate MCQs from
        num_mcqs           : Number of MCQs to generate (1-20)
        difficulty         : "Easy" | "Medium" | "Hard" | "Mixed"
        subject            : Subject/topic name
        include_explanation: Whether to include answer explanations
        api_key            : Groq API key

    Returns:
        List of dicts, each with: question, options (A/B/C/D), correct_answer, explanation
    """
    if not text.strip():
        raise ValueError("Input text is empty.")
    if not api_key.strip():
        raise ValueError("Groq API key is missing.")

    from groq import Groq # Import Groq client here to ensure it's available
    client = Groq(api_key=api_key.strip())

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile", # Ensure this model name is correct and available
            messages=[
                {"role": "system", "content": build_system_prompt(include_explanation)},
                {"role": "user",   "content": build_user_prompt(text, num_mcqs, difficulty, subject)}
            ],
            temperature=0.7,
            max_tokens=4096,
            response_format={"type": "json_object"}   # Groq JSON mode
        )
    except Exception as e:
        err = str(e)
        if "401" in err or "authentication" in err.lower():
            raise ValueError("Invalid API Key. Get yours at console.groq.com")
        elif "429" in err or "rate_limit" in err.lower():
            raise ValueError("Rate limit hit. Wait 60 seconds and try again.")
        else:
            raise ValueError(f"Groq API error: {err}")

    raw = response.choices[0].message.content
    if not raw:
        raise RuntimeError("Empty response from Groq API.")

    return parse_mcq_json(raw)


print("Core MCQ generation functions defined!")
print("Functions ready: generate_mcqs(), parse_mcq_json(), build_system_prompt(), build_user_prompt()")

Core MCQ generation functions defined!
Functions ready: generate_mcqs(), parse_mcq_json(), build_system_prompt(), build_user_prompt()


In [21]:
import textwrap # textwrap is needed for print_mcqs_plain

def print_mcqs_plain(mcqs: list, subject: str = ""):
    """Print MCQs in clean plain text format (no HTML)."""
    print("=" * 60)
    print(f"  MCQ SET — {subject.upper()}")
    print(f"  {len(mcqs)} Questions | Groq AI (llama-3.3-70b-versatile)")
    print("=" * 60)
    for i, mcq in enumerate(mcqs, 1):
        print(f"\nQ{i}. {mcq['question']}")
        for label in ["A", "B", "C", "D"]:
            opt = mcq["options"].get(label, "")
            if opt:
                marker = " ← CORRECT" if label == mcq["correct_answer"] else ""
                print(f"   {label}) {opt}{marker}")
        print(f"   Answer: {mcq['correct_answer']}")
        if mcq.get("explanation"):
            wrapped = textwrap.fill(mcq["explanation"], width=70, initial_indent="   💡 ", subsequent_indent="      ")
            print(wrapped)
    print("\n" + "=" * 60)


print("Display & formatting functions defined!")
print("   Functions ready: print_mcqs_plain()")

Display & formatting functions defined!
   Functions ready: print_mcqs_plain()


### Generate MCQs from the Extracted PDF Text

Now that the core functions are defined, you can generate MCQs directly from the `extracted_text` variable using the `generate_mcqs` function. Here's an example:

In [22]:
# Example usage of generate_mcqs

if 'extracted_text' in locals() and extracted_text:
    print("Generating MCQs...")
    try:
        generated_mcqs = generate_mcqs(
            text=extracted_text,
            num_mcqs=3,
            difficulty="Medium",
            subject="Big Data",
            include_explanation=True,
            api_key=GROQ_API_KEY
        )
        print_mcqs_plain(generated_mcqs, subject="Big Data")
    except Exception as e:
        print(f"Error generating MCQs: {e}")
else:
    print("No extracted text found. Please run the PDF upload and extraction cells first.")

Generating MCQs...
  MCQ SET — BIG DATA
  3 Questions | Groq AI (llama-3.3-70b-versatile)

Q1. What does the term 'Big Data' refer to?
   A) Small datasets that are easy to handle
   B) Extremely large datasets that are too complex for traditional data processing tools ← CORRECT
   C) Medium-sized datasets that can be handled by most tools
   D) Unstructured data only
   Answer: B
   💡 Big Data refers to extremely large datasets that are too complex
      and vast for traditional data processing tools to handle
      effectively.

Q2. What are the 5 V's that define the key characteristics of Big Data?
   A) Volume, Velocity, Variety, Veracity, Value ← CORRECT
   B) Speed, Size, Scope, Security, Storage
   C) Data, Database, Datamining, Datawarehousing, Dataanalysis
   D) Volume, Velocity, Variety, Validity, Visualization
   Answer: A
   💡 The 5 V's of Big Data are Volume, Velocity, Variety, Veracity,
      and Value, which define its key characteristics.

Q3. What is the primary purpos

### Gradio Interface for Interactive MCQ Generation

For a more interactive experience, you can use the Gradio interface to adjust parameters and generate MCQs directly from the uploaded PDF text.

In [ ]:
import gradio as gr
import io
from contextlib import redirect_stdout

# --- Gradio Interface Function ---
def gradio_generate_mcqs(num_mcqs: int, difficulty: str, subject: str, include_explanation: bool) -> str:
    global extracted_text, GROQ_API_KEY # Access global variables

    if not extracted_text:
        return "Error: No text extracted from PDF. Please ensure a PDF was uploaded and text extraction was successful."
    if not GROQ_API_KEY:
        return "Error: Groq API key is not set. Please ensure 'GROQ_API_KEY' is correctly configured in secrets."
    if not subject.strip():
        return "Error: Please provide a subject/topic for the MCQs."

    try:
        print(f"Generating {num_mcqs} {difficulty} MCQs on '{subject}'...")
        mcqs = generate_mcqs(
            text=extracted_text,
            num_mcqs=num_mcqs,
            difficulty=difficulty,
            subject=subject,
            include_explanation=include_explanation,
            api_key=GROQ_API_KEY
        )

        # Capture the output of print_mcqs_plain
        f = io.StringIO()
        with redirect_stdout(f):
            print_mcqs_plain(mcqs, subject=subject)
        output_str = f.getvalue()
        return output_str

    except ValueError as e:
        return f"Input Error: {e}"
    except RuntimeError as e:
        return f"Generation Error: {e}"
    except Exception as e:
        return f"An unexpected error occurred: {e}\n{traceback.format_exc()}" # Add traceback for debugging

# --- Gradio UI setup ---
print("Launching Gradio interface...")
iface = gr.Interface(
    fn=gradio_generate_mcqs,
    inputs=[
        gr.Slider(minimum=1, maximum=10, step=1, value=3, label="Number of MCQs"),
        gr.Dropdown(["Easy", "Medium", "Hard", "Mixed"], value="Medium", label="Difficulty"),
        gr.Textbox(lines=1, placeholder="e.g., Data Science, History, Biology", label="Subject/Topic"),
        gr.Checkbox(value=True, label="Include Explanations")
    ],
    outputs=gr.Markdown(), # Display output as Markdown for better formatting
    title="AI-Powered MCQ Generator from PDF",
    description="Generate multiple-choice questions from the uploaded PDF text using Groq's LLM."
)

# Launch the interface
iface.launch(debug=True, share=True)

Launching Gradio interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fbfde2d5d7802a67d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Generating 3 Medium MCQs on 'big data system'...


In [ ]:
def print_mcqs_plain(mcqs: list, subject: str = ""):
    """Print MCQs in clean plain text format (no HTML)."""
    print("=" * 60)
    print(f"  MCQ SET — {subject.upper()}")
    print(f"  {len(mcqs)} Questions | Groq AI (llama-3.3-70b-versatile)")
    print("=" * 60)
    for i, mcq in enumerate(mcqs, 1):
        print(f"\nQ{i}. {mcq['question']}")
        for label in ["A", "B", "C", "D"]:
            opt = mcq["options"].get(label, "")
            if opt:
                marker = " ← CORRECT" if label == mcq["correct_answer"] else ""
                print(f"   {label}) {opt}{marker}")
        print(f"   Answer: {mcq['correct_answer']}")
        if mcq.get("explanation"):
            wrapped = textwrap.fill(mcq["explanation"], width=70, initial_indent="   💡 ", subsequent_indent="      ")
            print(wrapped)
    print("\n" + "=" * 60)


print("Display & formatting functions defined!")
print("   Functions ready: display_mcqs(), print_mcqs_plain()")


In [ ]:
import os

# Create a directory to store the PDF document
if not os.path.exists('Docs'):
    os.makedirs('Docs')

print("Please upload your PDF file into the 'Docs/' folder in the file browser on the left.")

In [ ]:
!pip install pypdf
# Load the PDF document from the 'Docs' directory
loader = PyPDFDirectoryLoader('Docs/')
documents = loader.load()


print(f"Loaded {len(documents)} document(s).")
if documents:
    print("First 500 characters of the first document:")
    print(documents[0].page_content[:500])

### Split Documents into Chunks

To effectively use the document content for question generation, we need to split it into smaller, overlapping chunks. This helps the language model process information in manageable pieces.


In [ ]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split the documents
texts = text_splitter.split_documents(documents)

print(f"Split into {len(texts)} chunks.")
if texts:
    print("First chunk content:")
    print(texts[0].page_content[:500])

### Create Embeddings

Next, we'll create numerical representations (embeddings) of our text chunks using a HuggingFace embedding model. These embeddings will be stored in a vector store, allowing us to efficiently search and retrieve relevant text segments for question generation.

In [ ]:
# Initialize the HuggingFace embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("HuggingFace embeddings model initialized.")

In [ ]:
from google.colab import userdata
import os
from pinecone import Pinecone, ServerlessSpec, NotFoundException, PineconeApiException # Added NotFoundException, PineconeApiException

# Retrieve Pinecone API key and environment from Colab secrets
PINE_API_KEY = userdata.get('PINECONE_API_KEY')
PINE_ENV = userdata.get('PINECONE_ENVIRONMENT') # e.g., 'gcp-starter'

# Set environment variables for Pinecone
os.environ['PINECONE_API_KEY'] = PINE_API_KEY
os.environ['PINECONE_ENVIRONMENT'] = PINE_ENV

# Initialize Pinecone
pinecone = Pinecone(
    api_key=PINE_API_KEY,
    environment=PINE_ENV
)

INDEX_NAME = "test" # Using 'test' as suggested by previous error. Adjust if you have another preferred index.

# Define the expected dimension for the embeddings (from 'sentence-transformers/all-MiniLM-L6-v2')
EXPECTED_DIMENSION = 384

try:
    # Attempt to describe the index. If it fails, NotFoundException is raised.
    index_description = pinecone.describe_index(INDEX_NAME)

    # Index exists, check dimension
    if index_description.dimension != EXPECTED_DIMENSION:
        print(f"Index '{INDEX_NAME}' exists but has incorrect dimension ({index_description.dimension}). Deleting and recreating with dimension {EXPECTED_DIMENSION}.")
        pinecone.delete_index(INDEX_NAME)
        # Recreate after deletion
        pinecone.create_index(
            INDEX_NAME,
            dimension=EXPECTED_DIMENSION,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region=PINE_ENV)
        )
    else:
        print(f"Index '{INDEX_NAME}' already exists with correct dimension ({EXPECTED_DIMENSION}).")

except NotFoundException:
    # Index does not exist, attempt to create it
    print(f"Index '{INDEX_NAME}' not found. Attempting to create a new index with dimension {EXPECTED_DIMENSION}.")
    try:
        pinecone.create_index(
            INDEX_NAME,
            dimension=EXPECTED_DIMENSION, # Dimension of all-MiniLM-L6-v2 embeddings
            metric='cosine', # Common metric for embeddings
            spec=ServerlessSpec(cloud='aws', region=PINE_ENV) # Added spec argument
        )
        print(f"Successfully created index '{INDEX_NAME}'.")
    except PineconeApiException as e:
        # Specifically catch the ALREADY_EXISTS error during creation
        if e.status == 409 and "ALREADY_EXISTS" in str(e.body):
            print(f"Index '{INDEX_NAME}' was created during the process or already exists. Proceeding.")
            # Verify the dimension after assuming it exists
            try:
                index_description = pinecone.describe_index(INDEX_NAME)
                if index_description.dimension != EXPECTED_DIMENSION:
                    print(f"Warning: Index '{INDEX_NAME}' was created but has incorrect dimension ({index_description.dimension}). Please manually delete and recreate it with dimension {EXPECTED_DIMENSION}.")
            except NotFoundException:
                print(f"Warning: Index '{INDEX_NAME}' still not found after conflict. It might be in a transitional state.")
        else:
            raise e # Re-raise any other PineconeAPI exceptions

# Create a Pinecone vector store from the documents
docsearch = PineconeVectorStore.from_documents(texts, embeddings, index_name=INDEX_NAME)

print(f"Pinecone vector store '{INDEX_NAME}' created and loaded with {len(texts)} embeddings.")

In [ ]:
#This function will help us in fetching the top relevent documents from our vector store - Pinecone
def get_similiar_docs(query, k=2):
    similar_docs = index.similarity_search(query, k=k)
    return similar_docs